# Improve Connect 4 Model

This notebook allows you to continue training the Connect 4 AI agent to improve its performance.
It loads the existing model (`notebook_model.zip`), trains it against a Random Opponent for a specified number of steps, and saves it back.

In [ ]:
import sys
import os
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from pefforza.envs.connect4_env import Connect4Env

## Environment Wrapper
We need a wrapper to simulate the opponent's moves, so the PPO agent sees state -> action -> new state (after opponent).

In [ ]:
class SinglePlayerWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        
    def step(self, action):
        # 1. Agent Move
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        if terminated or truncated:
            return obs, reward, terminated, truncated, info
        
        # 2. Opponent Move (Random)
        # Find valid moves
        valid_moves = [c for c in range(self.env.cols) if self.env.board[0, c] == 0]
        if not valid_moves:
            return obs, 0, True, False, {} # Draw
            
        import random
        opp_action = random.choice(valid_moves)
        obs, reward, terminated, truncated, info = self.env.step(opp_action)
        
        # If Opponent Wins, Reward for Agent is -1
        # Connect4Env returns 1.0 if the *current* player won.
        # We just called step(opp_action), so if it returns 1.0, Opponent won.
        if terminated and reward == 1.0:
             reward = -1.0
        
        return obs, reward, terminated, truncated, info

## Configuration

In [ ]:
MODEL_PATH = os.path.join(project_root, "pefforza/agent/models/notebook_model.zip")
TIMESTEPS = 500000 # Number of steps to train. Increase for better results (e.g., 50000).

## Load and Train

In [ ]:
env = Connect4Env()
env = SinglePlayerWrapper(env)

if os.path.exists(MODEL_PATH):
    print(f"Loading model from {MODEL_PATH}...")
    model = PPO.load(MODEL_PATH, env=env)
else:
    print("Creating new model...")
    model = PPO("MlpPolicy", env, verbose=1)

print(f"Training for {TIMESTEPS} steps...")
model.learn(total_timesteps=TIMESTEPS)
print("Training complete.")

## Save Model

In [ ]:
model.save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")